In [1]:
class HiddenState:

    def __init__(self, name):
        self.name = name
        self.initial_prob = []
        self.transition_probs = []
        self.emission_probs = []

    # define how to print this object
    def __repr__(self):
        return (
            f"HiddenState('{self.name}')\n"
            f"  initial_prob:     {self.initial_prob}\n"
            f"  transition_probs: {self.transition_probs}\n"
            f"  emission_probs:   {self.emission_probs}\n"
        )

    # initial probabilities
    def add_initial(self, init_probs):
        self.initial_prob = init_probs[self.name]
    
    # transition probabilities
    def add_trans(self, trans_probs):
        self.transition_probs = trans_probs[self.name]

    # emission probabilities
    def add_emit(self, emit_probs):
        self.emission_probs = emit_probs[self.name]

    # qualify emissions (sum(all emissions for a state) = 1)
    def qualify_emissions(self):
        if sum(self.emission_probs.values()) == 1:
            print(f"Okay! Emission probabilities add to {sum(self.emission_probs.values())}")
        else:
            print(f"Emission probabilities add to {sum(self.emission_probs.values())}")

    

In [2]:
def build_state_dict(init_probs, trans_probs, emit_probs):
    '''Builds Hidden States Class Objects
    Params: obs (string): observation sequence
            init_probs (dict): initial probabilities of hidden states
            trans_probs (dict): transition probabilities
            emit_probs (dict): emission probabilities
    Return: state_dict (dict): dictionary of hidden states objects'''

    state_names = list(init_probs.keys())
    state_dict = {}

    for name in state_names:
        state = HiddenState(name)
        state.add_initial(init_probs)
        state.add_trans(trans_probs)
        state.add_emit(emit_probs)
        state_dict[name] = state

    return state_dict

In [3]:
# Function Viterbi Matrix:  

import numpy as np
from pprint import pprint
import math




def viterbi(states):
    '''Runs Viterbi algorithm. Builds array of probabilities based on observations as well as the traceback matrix which records where previous state was coming from.
    Params: states (dict): dict of state objects
    Return: prob_matrix (np.array): probability matrix
            traceback_matrix (np.array): traceback matrix'''

    #initialize
    prob_matrix = np.zeros((len(init_probs), len(obs)))
    traceback_matrix = np.zeros((len(init_probs), len(obs)))
    
    # make states
    # states = list(init_probs.keys())
    
    #for prev_state in states: print(prev_state)
    
    #iterate
    for i, observation in enumerate(obs):
        for j, state in enumerate(states):
            if i == 0:
                state_prob = states[state].initial_prob * states[state].emission_probs[observation]
                prob_matrix[j][i] = np.log(state_prob)
            else:            
                # for each state, calc prob
                possible_probs = [np.exp(prob_matrix[k][i-1]) * 
                                  states[prev_state].transition_probs[state] * 
                                  states[state].emission_probs[observation] 
                                  for k, prev_state in enumerate(states)
]
                #print(possible_probs)
                # argmax
                max_prob = max(possible_probs)
                #print(max_prob)
                prob_matrix[j][i] = np.log(max_prob)
                previous_coords = np.argmax(possible_probs)
                traceback_matrix[j][i] = previous_coords
    
    #pprint(prob_matrix)
    #pprint(traceback_matrix)

    return prob_matrix, traceback_matrix




In [4]:
# traceback

def traceback(obs, states, prob_matrix, traceback_matrix):
    '''Traces back through the traceback matrix to generate the hidden states sequence
    Params: obs (str): observation sequence
            states (dict): dictionary of state objects
            prob_matrix (np.array): probability matrix
            traceback_matrix (np.array): traceback matrix
    Return: states_final (list): hidden states sequence'''

    state_names = list(states.keys())
    # starting point is the highest probability for the final state in the sequence
    end_state = np.argmax(prob_matrix[:,-1])
    print(end_state)

    # trace backwards through traceback matrix and fill predictions list
    predictions = []
    predictions.append(end_state)
    for i in range(len(obs) -1, 0, -1):
        end_state = int(traceback_matrix[end_state][i])
        predictions.append(end_state)
        #print(end_state)

    predictions.reverse()
    
    print(predictions)
    
    # states_final = []
    # for observation in predictions:
    #     states_final.append(states[state_names[observation]])

    states_final = []
    for observation in predictions:
        state_name = state_names[observation]
        states_final.append(states[state_name].name)
    
    return states_final



In [ ]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [5]:
# Example observation sequence following the powerpoint
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.1,
    "G": 0.9
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.6, "G": 0.4},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4}
}

In [6]:
# driver
states = build_state_dict(init_probs, trans_probs, emit_probs)
prob_matrix, traceback_matrix = viterbi(states)
print(traceback(obs, states, prob_matrix, traceback_matrix))

1
[1, 0, 0, 0, 0, 1, 1, 1]
['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']


In [8]:
# optional: demonstrate print method for object
print(states)
print("\n")
print(states["G"])
states["G"].qualify_emissions()

{'I': HiddenState('I')
  initial_prob:     0.1
  transition_probs: {'I': 0.6, 'G': 0.4}
  emission_probs:   {'A': 0.1, 'C': 0.4, 'G': 0.4, 'T': 0.1}
, 'G': HiddenState('G')
  initial_prob:     0.9
  transition_probs: {'I': 0.1, 'G': 0.9}
  emission_probs:   {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}


HiddenState('G')
  initial_prob:     0.9
  transition_probs: {'I': 0.1, 'G': 0.9}
  emission_probs:   {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}

Okay! Emission probabilities add to 1.0
